In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)

RAW_PATH = "../data/raw/charging_record.csv"
CLEAN_PATH = "../data/processed/clean_sessions.csv"

raw = pd.read_csv(RAW_PATH)
df = raw.copy()          # work on a copy so the raw data is never changed

# A small log so we can report exactly what each step removed
cleaning_log = []

def record(step):
    """Save the number of rows left after a cleaning step."""
    cleaning_log.append({"step": step, "rows_left": len(df)})

record("Start (raw data)")
print(df.shape)

(72856, 13)


In [2]:
n_duplicates = df.duplicated().sum()
print("Exact duplicate rows:", n_duplicates)    # expect 13

df = df.drop_duplicates().reset_index(drop=True)
record("Removed duplicate rows")

Exact duplicate rows: 13


In [3]:
# Build full timestamps from the Day and Time columns (these keep the seconds)
df["StartDatetime"] = pd.to_datetime(df["StartDay"] + " " + df["StartTime"])
df["EndDatetime"] = pd.to_datetime(df["EndDay"] + " " + df["EndTime"])

# Day and Time columns are now redundant, so drop them
df = df.drop(columns=["StartDay", "StartTime", "EndDay", "EndTime"])

df[["StartDatetime", "EndDatetime"]].dtypes

StartDatetime    datetime64[us]
EndDatetime      datetime64[us]
dtype: object

In [4]:
ends_before_start = df["EndDatetime"] < df["StartDatetime"]
print("Sessions ending before they start:", ends_before_start.sum())   # expect 17

df = df[~ends_before_start].reset_index(drop=True)
record("Removed sessions that end before they start")

Sessions ending before they start: 17


In [5]:
# Duration in minutes, calculated from the timestamps
df["DurationCalc"] = (df["EndDatetime"] - df["StartDatetime"]).dt.total_seconds() / 60

# Flag rows where the recorded Duration disagrees by more than 5 minutes
df["DurationMismatch"] = (df["DurationCalc"] - df["Duration"]).abs() > 5

print("Rows where Duration disagrees:", df["DurationMismatch"].sum())   # about 2,200
df[df["DurationMismatch"]][
    ["StartDatetime", "EndDatetime", "Duration", "DurationCalc", "Demand"]
].head(8)

Rows where Duration disagrees: 2176


,StartDatetime,EndDatetime,Duration,DurationCalc,Demand
15,2022-08-19 17:10:57,2022-08-19 19:10:57,4,120.000000,12.42
50,2022-03-01 12:45:35,2022-03-01 13:34:38,0,49.050000,5.58
57,2022-03-13 09:49:06,2022-03-13 10:41:42,0,52.600000,4.66
66,2022-02-11 08:45:14,2022-02-11 10:00:14,0,75.000000,8.34
244,2021-12-26 16:34:37,2021-12-26 17:01:55,0,27.300000,3.03
246,2021-12-27 17:29:49,2021-12-27 20:24:22,0,174.550000,17.32
348,2022-03-31 22:46:37,2022-04-01 02:39:42,0,233.083333,16.87
349,2022-04-14 21:17:15,2022-04-15 00:29:33,0,192.300000,19.69


In [6]:
# Make Location text consistent (no stray spaces or capital letters)
df["Location"] = df["Location"].str.strip().str.lower()

# Readable labels for charts. The codes 0/1 are not defined in the brief,
# so we do NOT invent names like "fast charger".
df["ChargerCompanyLabel"] = "Company " + df["ChargerCompany"].astype(str)
df["ChargerTypeLabel"] = "Type " + df["ChargerType"].astype(str)

# UserID 0 looks like "unknown user", so remember which sessions have a real one
df["IsKnownUser"] = df["UserID"] != 0

df[["Location", "ChargerCompanyLabel", "ChargerTypeLabel", "IsKnownUser"]].head()

,Location,ChargerCompanyLabel,ChargerTypeLabel,IsKnownUser
0,hotel,Company 1,Type 0,False
1,hotel,Company 1,Type 0,False
2,hotel,Company 1,Type 0,False
3,hotel,Company 1,Type 0,False
4,hotel,Company 1,Type 0,False


In [7]:
def iqr_outliers(series):
    """Count values far outside the typical range (the IQR rule)."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((series < low) | (series > high)).sum()
    return count, round(low, 1), round(high, 1)

for column in ["Demand", "DurationCalc"]:
    count, low, high = iqr_outliers(df[column])
    print(f"{column}: {count} outliers outside [{low}, {high}]")

Demand: 3439 outliers outside [-16.0, 46.7]
DurationCalc: 4201 outliers outside [-204.5, 449.1]


In [8]:
# Time order matters later: history features and the train/test split need it
df = df.sort_values("StartDatetime").reset_index(drop=True)
record("Final clean data")
df.dtypes

UserID                          int64
ChargerID                       int64
ChargerCompany                  int64
Location                          str
ChargerType                     int64
StartDatetime          datetime64[us]
EndDatetime            datetime64[us]
Duration                        int64
Demand                        float64
DurationCalc                  float64
DurationMismatch                 bool
ChargerCompanyLabel               str
ChargerTypeLabel                  str
IsKnownUser                      bool
dtype: object

In [9]:
# assert stops the notebook with an error if a condition is False
assert df.duplicated().sum() == 0, "duplicates remain"
assert (df["EndDatetime"] >= df["StartDatetime"]).all(), "bad time order"
assert df.isna().sum().sum() == 0, "missing values appeared"
assert (df["Demand"] > 0).all(), "non-positive demand"
print("All checks passed")

All checks passed


In [10]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)

print("Saved:", CLEAN_PATH, df.shape)
pd.DataFrame(cleaning_log)

Saved: ../data/processed/clean_sessions.csv (72826, 14)


,step,rows_left
0,Start (raw data),72856
1,Removed duplicate rows,72843
2,Removed sessions that end before they start,72826
3,Final clean data,72826
